In [ ]:
# Colab setup: clone data assets and enable interactive widgets
import os

try:
    import google.colab
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab] Cloning repository assets...")
    # Clean up any leftover temporary directory before cloning
    !rm -rf _repo_tmp
    !git clone https://github.com/mattjunior039/SelfDrivingCar.git _repo_tmp
    !cp -r _repo_tmp/data . 2>/dev/null || true
    !rm -rf _repo_tmp
    
    # CRITICAL: Enable ipywidgets for the interactive tuning sliders
    output.enable_custom_widget_manager()
    
    print("[Colab] Ready and interactive widgets enabled.")

# Phase 3 — Real-Time Object Detection with YOLO
### Self-Driving Car Curriculum · Student Workbook

---

## Where we left off

Phase 2 ended with a wall. Your `MiniRoadCNN` could look at a 64×64 patch and say `"pedestrian"` — but point it at a real 1920×1080 dashcam frame and it produces three numbers and nothing else. No coordinates. No count. No sizes.

You traced the cause to a single line: `nn.Flatten()`. Up to that point the network still held a spatial grid, where cell $(4, 7)$ genuinely corresponded to a region of the image. Flattening poured that grid into a 1-D list and the next `Linear` layer blended every position together.

You also calculated why the obvious fix fails. Sliding a 64×64 classifier across a full frame at stride 16 needs roughly 7,000 windows per frame, times several scales, times 30 frames per second. Hundreds of thousands of forward passes every second, most of them recomputing near-identical convolutions on overlapping windows.

## The idea behind this phase

YOLO — **You Only Look Once** — resolves this with one architectural decision:

> **Don't flatten. Keep the spatial grid all the way to the output.**

Instead of collapsing to 3 class scores, the network outputs a *grid* of predictions. For a 640×640 input, one of YOLO's detection heads produces a 20×20 grid, and **every cell** predicts: is there an object near me, what class is it, and what are its box coordinates?

```
   PHASE 2 CLASSIFIER                    PHASE 3 DETECTOR
   ──────────────────                    ────────────────
   3×64×64                               3×640×640
      ↓ conv blocks                         ↓ conv blocks
   32×16×16   (grid intact)              256×20×20   (grid intact)
      ↓ FLATTEN  ← information dies         ↓ 1×1 conv  ← grid PRESERVED
   8192 vector                           (4+1+80) × 20 × 20
      ↓ Linear                              ↓
   3 numbers                             400 cells, each with a box + class
   "there is a car"                      "car at (812, 447), 96×61 px, conf 0.88"
```

The convolutional features are computed **once** for the whole image and shared across every grid cell. That is why YOLO runs at 100+ FPS while a sliding window struggles to hit 1.

## What this phase costs you

Keeping the grid creates a new problem. Grid cell $(9, 12)$ and grid cell $(9, 13)$ both sit near the same car, so both confidently report a box around it. A real frame with 8 vehicles might generate 300 raw boxes.

So detection needs two tools that classification never did:

1. **IoU** — a number that says how much two boxes overlap. Section 1.
2. **NMS** — an algorithm that uses IoU to delete duplicates. Section 2.

Neither is learned. Both are hand-tuned thresholds — which should feel familiar. Phase 1's brittleness never fully went away; it retreated into the post-processing.

## Learning objectives

By the end of this notebook you should be able to:
1. Compute IoU by hand and in code, including the edge case of non-overlapping boxes.
2. Explain why IoU is used instead of centre-distance or area difference.
3. Trace the NMS algorithm step by step and predict its output for a given threshold.
4. Explain the failure modes at both extremes of `iou_threshold`.
5. Run a pretrained YOLO model over a video and filter its output to road-relevant classes.
6. Identify temporal flickering, occlusion dropouts, and weather failures in real detector output — and explain why per-frame detection cannot fix them.

## Safety note

You are about to use a genuinely capable model. `yolov8n` is a real detector trained on 200,000 images, and it will feel impressive. Resist the impression. It is the *smallest* model in the family, trained on web photos rather than driving footage, with no depth estimate, no tracking, no motion prediction, and no notion of which detections matter. Section 4 exists to puncture the illusion. A perception stack that is safe to deploy involves redundant sensors, tracking, prediction, and validation over millions of miles.

---

## Setup

Sections 1 and 2 need only NumPy, Matplotlib, OpenCV, and `ipywidgets` — they will run anywhere.

Section 3 additionally needs `ultralytics`, which will download PyTorch if you do not have it. The first time you construct `YOLO('yolov8n.pt')` it downloads a ~6 MB weights file, so you need an internet connection once.

> **If you cannot install `ultralytics`**, Sections 1, 2, and 4 still work standalone, and Section 3 has a synthetic fallback so you can see the pipeline structure.

In [ ]:
# %pip install ultralytics opencv-python numpy matplotlib ipywidgets

import os
import time
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import ipywidgets as widgets
from IPython.display import display, clear_output, Video

plt.rcParams["figure.figsize"] = (10, 6)

try:
    from ultralytics import YOLO
    HAS_YOLO = True
except ImportError:
    HAS_YOLO = False

print("OpenCV     :", cv2.__version__)
print("ultralytics:", "available" if HAS_YOLO else "NOT installed (Section 3 will use the fallback)")

os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

---
# 1. Single-Stage Detection & IoU

## Box formats — read this before you debug anything

More detection bugs come from box format confusion than from anything else. There are three common conventions and they are trivially easy to mix up:

| Format | Contents | Used by |
|---|---|---|
| `xyxy` | `(x1, y1, x2, y2)` — top-left and bottom-right corners | YOLO output, most IoU code |
| `xywh` | `(x, y, width, height)` — top-left corner plus size | OpenCV, COCO annotations |
| `cxcywh` | `(cx, cy, width, height)` — **centre** plus size | YOLO's internal predictions |

Pass an `xywh` box to a function expecting `xyxy` and you will not get an error — you will get silently wrong overlaps. **This notebook uses `xyxy` everywhere.**

Also note the image coordinate convention: the origin is **top-left**, $x$ increases to the right, and $y$ increases **downward**. So `y1 < y2` always, which feels backwards if you are used to graph paper.

## Intersection over Union

You need a single number answering *"how much do these two boxes overlap?"* IoU is that number:

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

```
     ┌──────────────┐
     │  box A       │                       area of overlap
     │        ┌─────┼────────┐    IoU = ─────────────────────────
     │        │▒▒▒▒▒│        │           area covered by EITHER
     └────────┼─────┘        │
              │       box B  │
              └──────────────┘
```

IoU is always between 0 and 1:

| IoU | Meaning |
|---|---|
| `0.0` | no overlap at all |
| `0.3` | touching, but clearly different objects |
| `0.5` | the usual "correct detection" threshold in benchmarks |
| `0.75` | a tight, high-quality match |
| `1.0` | pixel-identical boxes |

## Why *this* formula?

Two tempting alternatives both fail:

- **Distance between centres.** A tiny box and a huge box sharing a centre would score as a perfect match. Size is ignored entirely.
- **Raw intersection area.** Two overlapping trucks share thousands of pixels; two overlapping phone-sized boxes share a handful. You could not use one threshold for both.

Dividing by the union fixes both. It makes the score **scale-invariant** — doubling both boxes leaves IoU unchanged — and it penalises a box that is too *large* as well as one that is too small, because excess area inflates the denominator.

## The computation

The intersection rectangle is found with a max/min trick:

```
ix1 = max(ax1, bx1)     ← the rightmost of the two left edges
iy1 = max(ay1, by1)     ← the lowest  of the two top edges
ix2 = min(ax2, bx2)     ← the leftmost of the two right edges
iy2 = min(ay2, by2)     ← the highest of the two bottom edges
```

> ### ⚠️ The clamping trap
> If the boxes do **not** overlap, you get `ix2 < ix1`, so the width `ix2 - ix1` comes out **negative**. Multiply a negative width by a negative height and you get a *positive* area — a completely phantom overlap between boxes on opposite sides of the image.
>
> This is one of the most common bugs in detection code, and it fails silently. Always clamp to zero:
> ```python
> inter_w = max(0.0, ix2 - ix1)
> inter_h = max(0.0, iy2 - iy1)
> ```

And the union uses inclusion–exclusion — add both areas, subtract the overlap you just double-counted:

$$\text{Union} = \text{Area}_A + \text{Area}_B - \text{Intersection}$$

### Exercise 1.1 — Implement IoU

Fill in the function. The test cases below will tell you immediately whether you got the clamping right.

In [ ]:
def box_area(box):
    """Area of an (x1, y1, x2, y2) box."""
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def compute_iou(box_a, box_b):
    """Intersection over Union of two (x1, y1, x2, y2) boxes."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # ==============================================================
    # YOUR CODE HERE
    #
    # 1. Intersection rectangle corners:
    #       ix1 = max(ax1, bx1)      iy1 = max(ay1, by1)
    #       ix2 = min(ax2, bx2)      iy2 = min(ay2, by2)
    #
    # 2. Width and height — CLAMP AT ZERO:
    #       inter_w = max(0.0, ix2 - ix1)
    #       inter_h = max(0.0, iy2 - iy1)
    #
    # 3. intersection = inter_w * inter_h
    #
    # 4. union = box_area(box_a) + box_area(box_b) - intersection
    #
    # 5. Return intersection / union, but guard against union == 0
    #    (two zero-size boxes) so you never divide by zero.
    # ==============================================================
    intersection = 0.0  # <-- replace
    union = 1.0         # <-- replace

    return intersection / union if union > 0 else 0.0

In [ ]:
TESTS = [
    ("identical boxes",        (0, 0, 10, 10), (0, 0, 10, 10),   1.0),
    ("no overlap (side)",      (0, 0, 10, 10), (20, 0, 30, 10),  0.0),
    ("no overlap (diagonal)",  (0, 0, 10, 10), (20, 20, 30, 30), 0.0),  # the clamping trap
    ("edges just touching",    (0, 0, 10, 10), (10, 0, 20, 10),  0.0),
    ("quarter overlap",        (0, 0, 10, 10), (5, 5, 15, 15),   25 / 175),
    ("half overlap",           (0, 0, 10, 10), (5, 0, 15, 10),   50 / 150),
    ("one fully inside other", (0, 0, 10, 10), (2, 2, 4, 4),     4 / 100),
]

print(f"{'case':<24} {'yours':>8} {'expected':>10}   result")
print("-" * 56)
all_pass = True
for name, a, b, expected in TESTS:
    got = compute_iou(a, b)
    ok = abs(got - expected) < 1e-6
    all_pass &= ok
    print(f"{name:<24} {got:>8.4f} {expected:>10.4f}   {'pass' if ok else 'FAIL'}")

print("\nAll tests passed." if all_pass else
      "\nSome tests failed. The diagonal case failing means you skipped the clamping step.")

### Exercise 1.2 — The interactive IoU explorer

Drag the sliders to move and resize the two boxes. The overlap region, the IoU value, and the interpretation bar all update live.

**Experiments to run, in order:**
1. Line the boxes up **exactly** — confirm IoU hits 1.00. Now nudge one by a few pixels. Notice how gently it falls: IoU is forgiving of small errors, which is what you want from a detection metric.
2. Shrink box B to be **tiny and fully inside** box A. IoU collapses toward 0 even though B is 100% contained. Ask yourself why — and what that means for detecting a child standing in front of a truck.
3. Separate the boxes completely, then drag them **diagonally** apart. If your clamping is right this stays at 0.00.
4. Find the configurations that give exactly **0.5** — the standard benchmark threshold. Are those two boxes visibly "the same object" to your eye? Several very different-looking arrangements all score 0.5.
5. Make the boxes **identical in size but offset**. How far can you push them before IoU drops under 0.5?

In [ ]:
CANVAS = 100


def iou_explorer(ax1, ay1, a_w, a_h, bx1, by1, b_w, b_h):
    box_a = (ax1, ay1, ax1 + a_w, ay1 + a_h)
    box_b = (bx1, by1, bx1 + b_w, by1 + b_h)
    iou = compute_iou(box_a, box_b)

    ix1, iy1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    ix2, iy2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    inter_w, inter_h = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    intersection = inter_w * inter_h
    union = box_area(box_a) + box_area(box_b) - intersection

    fig, (ax, bar) = plt.subplots(
        1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [2, 1]}
    )

    ax.add_patch(mpatches.Rectangle((box_a[0], box_a[1]), a_w, a_h,
                                    fill=True, facecolor="tab:blue", alpha=0.22,
                                    edgecolor="tab:blue", lw=2.5, label="Box A"))
    ax.add_patch(mpatches.Rectangle((box_b[0], box_b[1]), b_w, b_h,
                                    fill=True, facecolor="tab:orange", alpha=0.22,
                                    edgecolor="tab:orange", lw=2.5, label="Box B"))

    if intersection > 0:
        ax.add_patch(mpatches.Rectangle((ix1, iy1), inter_w, inter_h,
                                        fill=True, facecolor="tab:red", alpha=0.55,
                                        edgecolor="darkred", lw=2, hatch="//",
                                        label="Intersection"))

    ax.set_xlim(0, CANVAS)
    ax.set_ylim(CANVAS, 0)  # flipped: image coordinates put y=0 at the top
    ax.set_aspect("equal")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"IoU = {iou:.4f}", fontsize=17, fontweight="bold")
    ax.set_xlabel("x  (increases right)")
    ax.set_ylabel("y  (increases DOWN)")

    color = "tab:red" if iou < 0.3 else ("tab:orange" if iou < 0.5 else "tab:green")
    bar.barh([0], [iou], color=color, height=0.5)
    bar.barh([0], [1.0], color="lightgray", height=0.5, zorder=0)
    bar.axvline(0.5, ls="--", c="k", lw=1.5)
    bar.text(0.5, 0.42, " benchmark 0.5", fontsize=9)
    bar.set_xlim(0, 1)
    bar.set_ylim(-0.6, 0.8)
    bar.set_yticks([])
    bar.set_xlabel("IoU")
    bar.set_title("overlap quality")

    plt.tight_layout()
    plt.show()

    print(f"  Box A {box_a}   area = {box_area(box_a):8.1f}")
    print(f"  Box B {box_b}   area = {box_area(box_b):8.1f}")
    print(f"  Intersection = {intersection:8.1f}")
    print(f"  Union        = {union:8.1f}   ({box_area(box_a):.0f} + {box_area(box_b):.0f} - {intersection:.0f})")
    print(f"  IoU          = {intersection:.1f} / {union:.1f} = {iou:.4f}")


def _s(value, description, lo=0, hi=CANVAS):
    return widgets.IntSlider(value=value, min=lo, max=hi, step=1,
                             description=description, continuous_update=False)


explorer = widgets.interactive(
    iou_explorer,
    ax1=_s(20, "A: x"), ay1=_s(20, "A: y"),
    a_w=_s(40, "A: width", 1, CANVAS), a_h=_s(40, "A: height", 1, CANVAS),
    bx1=_s(40, "B: x"), by1=_s(40, "B: y"),
    b_w=_s(40, "B: width", 1, CANVAS), b_h=_s(40, "B: height", 1, CANVAS),
)

display(widgets.VBox([
    widgets.HBox([
        widgets.VBox(explorer.children[0:4]),
        widgets.VBox(explorer.children[4:8]),
    ]),
    explorer.children[-1],
]))
explorer.update()

> ### ✍️ Reflection 1
>
> 1. In experiment 2 you put a small box entirely inside a large one and IoU still came out low. Explain using the formula. Is "low overlap" the right verdict there? Describe a driving scenario where this specific behaviour would be dangerous.
> 2. IoU is scale-invariant: a 10×10 and a 20×20 pair of boxes can score the same as a 100×100 and 200×200 pair. Why is that a desirable property for a detector that must handle both nearby and distant cars?
> 3. You found several different arrangements that all score exactly 0.5. What does that tell you about compressing a 2-D spatial relationship into a single number? What information is lost?

*Your answers:*

1. 
2. 
3. 

---
# 2. Non-Maximum Suppression (NMS) Tuning

## The duplicate problem

Here is the consequence of keeping the spatial grid. A car occupies a region of the image that spans several grid cells, and **every one of those cells** predicts a box. Add multiple detection scales and multiple anchor shapes, and a single car can generate dozens of overlapping boxes.

Raw YOLO output for a typical street scene is on the order of **8,400 boxes**. Most have near-zero confidence and are dropped immediately by a confidence threshold, but that still leaves clusters of 5–20 boxes hugging each real object.

A downstream planner asking "how many cars are ahead?" must get `3`, not `47`.

## The algorithm

NMS is greedy and short enough to hold in your head:

```
  1. Sort every box by confidence, highest first.
  2. Take the highest-confidence box. KEEP it.
  3. Compare it against every remaining box.
     Any box with IoU > threshold is a duplicate → DISCARD it.
  4. Go back to step 2 with whatever survives.
  5. Stop when nothing is left.
```

The name says exactly what it does: **suppress** anything that is **not** the local **maximum** in confidence.

```
  before NMS                    after NMS
  ┌─┬┬─────┐                    ┌───────┐
  ├─┴┤ car │   0.91             │  car  │  0.91
  │  │ car │   0.87  ──────▶    │       │
  │  │ car │   0.84             │       │   the 0.87 and 0.84 boxes
  └──┴─────┘                    └───────┘   overlapped it by > threshold
```

## Per-class NMS

One subtlety: NMS is normally run **separately for each class**. A pedestrian standing directly in front of a bus produces heavily overlapping boxes, but they are different objects and both must survive. Suppressing across classes would delete the pedestrian. You will see this handled explicitly in the code.

## The threshold is a trade-off with no right answer

| `iou_threshold` | Behaviour | Failure |
|---|---|---|
| **0.1 (aggressive)** | almost anything touching gets suppressed | two cars side by side in traffic merge into **one** — the car plans a path through a vehicle it deleted |
| **0.45 (typical)** | the usual default | works most of the time |
| **0.9 (permissive)** | only near-identical boxes suppressed | duplicates survive — one pedestrian reported as four, and the planner cannot count |

Notice what this is: a hand-tuned number, chosen by a human, that trades one kind of failure against another. **Exactly like Phase 1's HSV thresholds.** The learning fixed the feature extraction; it did not eliminate hand-tuning, it relocated it.

### Exercise 2.1 — Build a scene with duplicates

We create a street-like image and place three real objects in it, then simulate raw detector output by scattering jittered duplicate boxes around each one — exactly the mess NMS has to clean up.

If you have a real street photo, point `STREET_IMAGE_PATH` at it.

In [ ]:
STREET_IMAGE_PATH = "data/street.jpg"
W, H = 640, 400


def make_street_scene():
    """Synthetic road scene: sky, asphalt, lane dashes, two cars and a pedestrian."""
    img = np.zeros((H, W, 3), dtype=np.uint8)
    img[: H // 2] = (150, 180, 210)   # sky
    img[H // 2 :] = (95, 95, 100)     # asphalt
    for x in range(0, W, 70):         # lane dashes
        cv2.rectangle(img, (x, 330), (x + 38, 338), (235, 235, 235), -1)

    cv2.rectangle(img, (70, 215), (215, 295), (40, 60, 170), -1)     # red car
    cv2.rectangle(img, (92, 225), (193, 255), (25, 30, 45), -1)
    cv2.circle(img, (105, 295), 13, (18, 18, 18), -1)
    cv2.circle(img, (185, 295), 13, (18, 18, 18), -1)

    cv2.rectangle(img, (330, 230), (450, 292), (150, 140, 60), -1)   # blue car
    cv2.rectangle(img, (348, 238), (432, 262), (25, 30, 45), -1)
    cv2.circle(img, (358, 292), 11, (18, 18, 18), -1)
    cv2.circle(img, (424, 292), 11, (18, 18, 18), -1)

    cv2.circle(img, (545, 205), 11, (150, 175, 205), -1)             # pedestrian
    cv2.rectangle(img, (534, 218), (557, 268), (60, 120, 60), -1)
    cv2.line(img, (541, 268), (537, 300), (45, 45, 90), 6)
    cv2.line(img, (550, 268), (555, 300), (45, 45, 90), 6)
    return img


_loaded = cv2.imread(STREET_IMAGE_PATH, cv2.IMREAD_COLOR)
if _loaded is None:
    print(f"'{STREET_IMAGE_PATH}' not found — using a synthetic street scene.")
    street_bgr = make_street_scene()
else:
    street_bgr = cv2.resize(_loaded, (W, H), interpolation=cv2.INTER_AREA)

street_rgb = cv2.cvtColor(street_bgr, cv2.COLOR_BGR2RGB)

GROUND_TRUTH = [
    {"box": (70, 215, 215, 295), "cls": "car"},
    {"box": (330, 230, 450, 292), "cls": "car"},
    {"box": (528, 192, 563, 302), "cls": "person"},
]


def scatter_duplicates(truth, n_dupes=6, jitter=22, seed=7):
    """Simulate raw detector output: many jittered, varyingly-confident boxes per object."""
    rng = np.random.default_rng(seed)
    raw = []
    for obj in truth:
        x1, y1, x2, y2 = obj["box"]
        raw.append({"box": (float(x1), float(y1), float(x2), float(y2)),
                    "score": float(rng.uniform(0.88, 0.97)), "cls": obj["cls"]})
        for _ in range(n_dupes):
            dx, dy = rng.normal(0, jitter, 2)
            ds = rng.normal(0, jitter * 0.6, 2)
            raw.append({
                "box": (float(x1 + dx - ds[0]), float(y1 + dy - ds[1]),
                        float(x2 + dx + ds[0]), float(y2 + dy + ds[1])),
                "score": float(rng.uniform(0.35, 0.87)),
                "cls": obj["cls"],
            })
    rng.shuffle(raw)
    return raw


raw_detections = scatter_duplicates(GROUND_TRUTH)
print(f"{len(raw_detections)} raw boxes for {len(GROUND_TRUTH)} real objects.")
print("Class counts:", Counter(d["cls"] for d in raw_detections))

### Exercise 2.2 — Implement NMS

Translate the five-step algorithm above into code. The `keep` list collects survivors; the `while` loop consumes the sorted list.

**Hint on structure:** pop the first (highest-confidence) item, append it to `keep`, then rebuild the remaining list keeping only boxes whose IoU with it is `<=` the threshold.

In [ ]:
def nms_single_class(detections, iou_threshold):
    """Greedy NMS over detections that all share one class."""
    # Step 1: sort by confidence, highest first.
    remaining = sorted(detections, key=lambda d: -d["score"])
    keep = []

    while remaining:
        # ==============================================================
        # YOUR CODE HERE
        #
        # Step 2: take the highest-confidence box off the front and keep it.
        #     best = remaining.pop(0)
        #     keep.append(best)
        #
        # Step 3: discard everything that overlaps `best` too much.
        #     remaining = [
        #         d for d in remaining
        #         if compute_iou(best["box"], d["box"]) <= iou_threshold
        #     ]
        #
        # Step 4 happens automatically — the while loop repeats.
        #
        # Delete the `break` once you have written the two steps above,
        # otherwise this loop exits after one pass.
        # ==============================================================
        break

    return keep


def nms(detections, iou_threshold):
    """Run NMS independently per class, so a person in front of a bus survives."""
    by_class = defaultdict(list)
    for d in detections:
        by_class[d["cls"]].append(d)

    kept = []
    for class_dets in by_class.values():
        kept.extend(nms_single_class(class_dets, iou_threshold))

    return sorted(kept, key=lambda d: -d["score"])


_result = nms(raw_detections, 0.45)
print(f"{len(raw_detections)} boxes -> {len(_result)} after NMS at threshold 0.45")
print(f"(There are {len(GROUND_TRUTH)} real objects.)")
if len(_result) == len(raw_detections):
    print("\nNothing was suppressed — did you remove the `break`?")

### Exercise 2.3 — The interactive NMS tuner

Drag the threshold and watch boxes appear and vanish.

**Experiments:**
1. Set `iou_threshold = 0.05`. Count the surviving boxes. Did the two cars collapse into one detection? Look at what got deleted.
2. Set `iou_threshold = 0.95`. Now almost every duplicate survives. Imagine a planner trying to count pedestrians from this.
3. Sweep slowly from 0.2 to 0.7 and find where the count first equals **3**. That is the correct answer *for this scene*. Would that same value work on a crowded crosswalk?
4. Raise `confidence_threshold` to 0.8 first, then sweep. Confidence filtering does some of NMS's work for free — but notice which real objects you lose.
5. Turn on `show_suppressed` to see deleted boxes as faint dashed outlines. This is the clearest view of what the algorithm is actually doing.

In [ ]:
CLASS_COLORS = {"car": "tab:cyan", "person": "yellow", "truck": "tab:orange"}


def nms_tuner(iou_threshold, confidence_threshold, show_suppressed):
    candidates = [d for d in raw_detections if d["score"] >= confidence_threshold]
    kept = nms(candidates, iou_threshold)
    kept_ids = {id(d) for d in kept}
    suppressed = [d for d in candidates if id(d) not in kept_ids]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].imshow(street_rgb)
    for d in candidates:
        x1, y1, x2, y2 = d["box"]
        axes[0].add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, fill=False,
            edgecolor=CLASS_COLORS.get(d["cls"], "white"), lw=1.2, alpha=0.75))
    axes[0].set_title(f"BEFORE NMS — {len(candidates)} boxes", fontsize=12)

    axes[1].imshow(street_rgb)
    if show_suppressed:
        for d in suppressed:
            x1, y1, x2, y2 = d["box"]
            axes[1].add_patch(mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1, fill=False,
                edgecolor="red", lw=0.9, ls=":", alpha=0.45))
    for d in kept:
        x1, y1, x2, y2 = d["box"]
        color = CLASS_COLORS.get(d["cls"], "white")
        axes[1].add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, lw=2.8))
        axes[1].text(x1, y1 - 5, f"{d['cls']} {d['score']:.2f}", color="black",
                     fontsize=9, bbox=dict(facecolor=color, alpha=0.85, pad=1.5))

    n_real = len(GROUND_TRUTH)
    verdict = ("correct count" if len(kept) == n_real else
               "OVER-counting (duplicates survived)" if len(kept) > n_real else
               "UNDER-counting (real objects deleted)")
    axes[1].set_title(f"AFTER NMS @ {iou_threshold:.2f} — {len(kept)} boxes · {verdict}",
                      fontsize=12)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"  raw {len(raw_detections)}  →  confidence≥{confidence_threshold:.2f}: "
          f"{len(candidates)}  →  NMS@{iou_threshold:.2f}: {len(kept)}")
    print(f"  kept by class: {dict(Counter(d['cls'] for d in kept))}")
    print(f"  ground truth : {dict(Counter(o['cls'] for o in GROUND_TRUTH))}")


tuner = widgets.interactive(
    nms_tuner,
    iou_threshold=widgets.FloatSlider(value=0.45, min=0.01, max=0.99, step=0.01,
                                      description="iou_thresh", continuous_update=False,
                                      readout_format=".2f"),
    confidence_threshold=widgets.FloatSlider(value=0.30, min=0.0, max=0.99, step=0.01,
                                             description="min conf", continuous_update=False,
                                             readout_format=".2f"),
    show_suppressed=widgets.Checkbox(value=True, description="show suppressed"),
)
display(tuner)

### Exercise 2.4 — Sweep the threshold

Rather than eyeballing it, plot the box count across the whole threshold range. The correct answer is a *plateau*, not a point — and noticing how narrow that plateau is tells you how fragile the tuning is.

In [ ]:
thresholds = np.arange(0.02, 1.0, 0.02)
counts = [len(nms(raw_detections, float(t))) for t in thresholds]

plt.figure(figsize=(11, 4.5))
plt.plot(thresholds, counts, "o-", ms=3.5)
plt.axhline(len(GROUND_TRUTH), ls="--", c="green",
            label=f"correct answer ({len(GROUND_TRUTH)} objects)")
plt.axvline(0.45, ls=":", c="gray", label="typical default (0.45)")
plt.fill_between(thresholds, 0, counts, where=np.array(counts) < len(GROUND_TRUTH),
                 color="red", alpha=0.12, label="under-counting (objects deleted)")
plt.xlabel("iou_threshold")
plt.ylabel("boxes surviving NMS")
plt.title("How many detections survive, as a function of the threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

good = [f"{t:.2f}" for t, c in zip(thresholds, counts) if c == len(GROUND_TRUTH)]
print("Thresholds giving the correct count:", ", ".join(good) if good else "none")
print("\nThat window is the entire margin for error — on ONE scene, with ONE arrangement")
print("of objects. A different scene moves the window.")

> ### ✍️ Reflection 2
>
> 1. At a very low threshold the two cars merged into one detection. Walk through the algorithm and explain precisely which step deleted the second car.
> 2. Why must NMS run per class? Give a concrete street scenario where cross-class suppression would remove a safety-critical detection.
> 3. NMS keeps the box with the highest *confidence*. But confidence measures how sure the model is about the **class**, not how accurate the **coordinates** are. Explain how this could keep a poorly-positioned box and discard a well-positioned one.
> 4. Compare `iou_threshold` to Phase 1's `s_min` slider. In what sense did the CNN not actually eliminate hand-tuning?

*Your answers:*

1. 
2. 
3. 
4. 

---
# 3. Real-Time Inference Pipeline

## What you are about to run

`yolov8n` (n for *nano*) is the smallest model in the YOLOv8 family: about 3.2 million parameters — roughly six times your `MiniRoadCNN` — trained on **COCO**, a dataset of 200,000 everyday photos across 80 classes.

Everything you built by hand in Sections 1 and 2 is already inside it. The `conf` and `iou` arguments you are about to pass *are* the confidence threshold and the NMS threshold you just tuned.

## COCO classes relevant to driving

COCO has 80 classes, most of them irrelevant on a road (`toaster`, `teddy bear`, `broccoli`). Filtering to the ones that matter reduces clutter and prevents a planner from reacting to a detected `potted plant`:

| ID | Class | ID | Class |
|---|---|---|---|
| 0 | person | 5 | bus |
| 1 | bicycle | 7 | truck |
| 2 | car | 9 | traffic light |
| 3 | motorcycle | 11 | stop sign |

> **Notice what COCO does *not* have:** lane markings, potholes, road debris, construction cones, emergency vehicles, crosswalks, or gestures from a traffic officer. A model can only detect categories someone thought to label. This is Phase 1's coverage problem wearing a new outfit.

## The video loop

```
  cap = cv2.VideoCapture(path)
  writer = cv2.VideoWriter(out, fourcc, fps, (w, h))
  while True:
      ok, frame = cap.read()          ← BGR, one frame
      if not ok: break                ← end of video
      results = model(frame)          ← detect
      ... draw boxes ...
      writer.write(frame)             ← must match (w, h) EXACTLY
  cap.release(); writer.release()     ← ALWAYS release
```

### Three things that will bite you

1. **`VideoWriter` size mismatch.** If the frame you write is not exactly the `(width, height)` you gave the constructor, it fails **silently** and produces a 0-byte or unplayable file. No exception. If your output won't open, check this first.
2. **Forgetting `release()`.** The file stays buffered and incomplete. Always use `try/finally`.
3. **Codec availability.** `mp4v` is the most portable choice. `avc1`/`H264` gives better browser playback but is not always installed.

## Getting a video

Put a short dashcam-style clip at `data/drive.mp4` — 10–30 seconds is plenty. Phone footage through a windscreen works well. If no file is found, the next cell synthesises a simple driving clip so you can still run the pipeline.

In [ ]:
INPUT_VIDEO = "data/drive.mp4"
OUTPUT_VIDEO = "output/detected.mp4"


def synthesize_drive_clip(path, n_frames=150, fps=25):
    """Fallback clip: a car approaching and a pedestrian crossing, with an occlusion event."""
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    for i in range(n_frames):
        f = make_street_scene()
        t = i / n_frames

        scale = 0.5 + 1.3 * t                      # car grows as it approaches
        cw, ch = int(90 * scale), int(55 * scale)
        cx, cy = int(150 + 90 * t), int(250 + 40 * t)
        cv2.rectangle(f, (cx, cy), (cx + cw, cy + ch), (40, 60, 170), -1)
        cv2.rectangle(f, (cx + 8, cy + 6), (cx + cw - 8, cy + ch // 2), (25, 30, 45), -1)

        px = int(600 - 420 * t)                    # pedestrian crossing right to left
        cv2.circle(f, (px, 205), 11, (150, 175, 205), -1)
        cv2.rectangle(f, (px - 11, 218), (px + 11, 268), (60, 120, 60), -1)

        if 0.45 < t < 0.6:                         # a pole briefly occludes the pedestrian
            cv2.rectangle(f, (px - 16, 150), (px + 16, 330), (70, 70, 75), -1)

        writer.write(f)
    writer.release()


if not os.path.exists(INPUT_VIDEO):
    print(f"'{INPUT_VIDEO}' not found — synthesizing a fallback clip.")
    print("NOTE: YOLO is trained on real photos and will detect little in synthetic video.")
    print("      Use real dashcam footage for Section 4 to be meaningful.\n")
    synthesize_drive_clip(INPUT_VIDEO)

probe = cv2.VideoCapture(INPUT_VIDEO)
src_w = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
src_fps = probe.get(cv2.CAP_PROP_FPS) or 25.0
src_n = int(probe.get(cv2.CAP_PROP_FRAME_COUNT))
ok, first_frame = probe.read()
probe.release()

print(f"Resolution : {src_w} x {src_h}")
print(f"FPS        : {src_fps:.2f}")
print(f"Frames     : {src_n}  (~{src_n / max(src_fps, 1):.1f}s)")

if ok:
    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    plt.title("First frame")
    plt.axis("off")
    plt.show()

### Exercise 3.1 — Load the model and test on one frame

Always debug on a single frame before committing to a whole video. A bug you catch here costs seconds; the same bug found after processing 900 frames costs minutes each time you retry.

In [ ]:
# COCO class IDs that matter on a road.
ROAD_CLASS_IDS = [0, 1, 2, 3, 5, 7, 9, 11]

CONF_THRESHOLD = 0.35   # the confidence slider from Section 2
IOU_THRESHOLD = 0.45    # the NMS slider from Section 2

# ------------------------------------------------------------------
# YOUR CODE HERE
# Instantiate the pretrained nano model:
#     model = YOLO("yolov8n.pt")
# The first call downloads ~6 MB of weights.
# ------------------------------------------------------------------
model = None  # <-- replace


if HAS_YOLO and model is not None:
    print("Model loaded.")
    print("Filtering to:", [model.names[i] for i in ROAD_CLASS_IDS])

    results = model(first_frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                    classes=ROAD_CLASS_IDS, verbose=False)

    boxes = results[0].boxes
    print(f"\n{len(boxes)} detections on frame 1:")
    for b in boxes:
        cls_id = int(b.cls[0])
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        print(f"  {model.names[cls_id]:<14} conf {float(b.conf[0]):.2f}  "
              f"box ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
    plt.title("YOLOv8n on a single frame")
    plt.axis("off")
    plt.show()
else:
    print("Model not loaded. Fill in the YOLO(...) line above"
          + ("" if HAS_YOLO else ", and install ultralytics") + ".")

### Exercise 3.2 — The full video loop

Complete the four marked blocks. The surrounding scaffolding — capture, writer, timing, statistics, cleanup — is written for you.

We draw boxes manually rather than using `results.plot()` so that every step is visible and editable. Note the label-background rectangle drawn before the text: without it, white text on a white truck is invisible.

> **Speed tip:** set `FRAME_STRIDE = 2` to process every other frame while you are debugging. Set it back to 1 for your final run.

In [ ]:
FRAME_STRIDE = 1
MAX_FRAMES = None  # set to e.g. 120 to cut a long clip short

BOX_COLORS = {
    0: (0, 255, 255), 1: (255, 200, 0), 2: (0, 255, 0), 3: (255, 100, 255),
    5: (255, 160, 0), 7: (0, 160, 255), 9: (200, 0, 255), 11: (0, 0, 255),
}

cap = cv2.VideoCapture(INPUT_VIDEO)
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    src_fps / FRAME_STRIDE,
    (src_w, src_h),  # MUST match the frames you write, exactly
)

frame_idx, processed = 0, 0
per_frame_counts, inference_times = [], []
start = time.time()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1
        if frame_idx % FRAME_STRIDE != 0:
            continue
        if MAX_FRAMES and processed >= MAX_FRAMES:
            break

        t0 = time.time()

        # ==============================================================
        # YOUR CODE HERE (1 of 4) — run detection on this frame
        #
        #   results = model(frame,
        #                   conf=CONF_THRESHOLD,
        #                   iou=IOU_THRESHOLD,
        #                   classes=ROAD_CLASS_IDS,
        #                   verbose=False)
        #
        # `classes=` does the road-relevant filtering for you.
        # `verbose=False` stops it printing a line per frame.
        # ==============================================================
        results = None  # <-- replace

        inference_times.append(time.time() - t0)

        # ==============================================================
        # YOUR CODE HERE (2 of 4) — pull out the boxes
        #
        #   detections = results[0].boxes
        # ==============================================================
        detections = []  # <-- replace

        counts = Counter()
        for det in detections:
            cls_id = int(det.cls[0])
            conf = float(det.conf[0])
            x1, y1, x2, y2 = (int(v) for v in det.xyxy[0].tolist())
            label = model.names[cls_id]
            counts[label] += 1
            color = BOX_COLORS.get(cls_id, (255, 255, 255))

            # ==============================================================
            # YOUR CODE HERE (3 of 4) — draw the box and its label
            #
            #   cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            #
            #   text = f"{label} {conf:.2f}"
            #   (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            #   cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
            #   cv2.putText(frame, text, (x1 + 2, y1 - 4),
            #               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1,
            #               cv2.LINE_AA)
            # ==============================================================
            pass

        per_frame_counts.append(counts)

        summary = "  ".join(f"{k}:{v}" for k, v in sorted(counts.items())) or "nothing detected"
        cv2.putText(frame, f"frame {frame_idx}   {summary}", (10, 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

        # ==============================================================
        # YOUR CODE HERE (4 of 4) — write the annotated frame
        #
        #   writer.write(frame)
        # ==============================================================
        pass

        processed += 1
        if processed % 25 == 0:
            print(f"  ...{processed} frames")

finally:
    cap.release()
    writer.release()

elapsed = time.time() - start
print(f"\nProcessed {processed} frames in {elapsed:.1f}s")

if inference_times and processed:
    mean_ms = 1000 * np.mean(inference_times)
    print(f"Mean inference: {mean_ms:.1f} ms/frame  ->  {1000 / max(mean_ms, 1e-6):.1f} FPS")
    print(f"Source video is {src_fps:.0f} FPS, so this is "
          f"{'FAST ENOUGH for real time' if 1000 / max(mean_ms, 1e-6) >= src_fps else 'TOO SLOW for real time'}.")

size = os.path.getsize(OUTPUT_VIDEO) if os.path.exists(OUTPUT_VIDEO) else 0
print(f"\nOutput: {OUTPUT_VIDEO}  ({size / 1e6:.2f} MB)")
if size < 10000:
    print("That file is suspiciously small — check that you completed block 4,")
    print("and that the frames you write match (src_w, src_h) exactly.")

In [ ]:
# Play the result inline. If it does not render, open output/detected.mp4 directly.
Video(OUTPUT_VIDEO, embed=False, width=720) if os.path.exists(OUTPUT_VIDEO) else print("No output yet.")

### Exercise 3.3 — Plot detections over time

This chart is the most important diagnostic in the notebook, and it is the one nobody thinks to make.

The scene changes smoothly — cars do not teleport, pedestrians do not evaporate. So the detection count *should* be a smooth curve. **Every vertical spike is a frame where the model changed its mind about reality.**

In [ ]:
if per_frame_counts:
    labels = sorted({k for c in per_frame_counts for k in c})
    frames = np.arange(1, len(per_frame_counts) + 1)

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

    for lab in labels:
        axes[0].plot(frames, [c.get(lab, 0) for c in per_frame_counts], label=lab, lw=1.4)
    axes[0].set_ylabel("count")
    axes[0].set_title("Detections per class, per frame — every jump is an instability")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    totals = np.array([sum(c.values()) for c in per_frame_counts])
    axes[1].plot(frames, totals, "k-", lw=1.4)
    changes = np.where(np.diff(totals) != 0)[0] + 1
    axes[1].vlines(changes, totals.min(), totals.max(), color="red", alpha=0.22, lw=1)
    axes[1].set_xlabel("frame")
    axes[1].set_ylabel("total objects")
    axes[1].set_title(f"Total count — it changed on {len(changes)} of {len(totals)} frames")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    flicker = 100 * len(changes) / max(len(totals) - 1, 1)
    print(f"The object count changed on {flicker:.1f}% of frame transitions.")
    print("Physical objects do not appear and disappear. Those are detector errors.")
else:
    print("Run Exercise 3.2 first.")

---
# 4. Edge-Case Evaluation

## The detector is good. That is the problem.

YOLO works well enough to be seductive. Boxes snap onto cars, labels look right, and it feels solved. Your job in this section is to watch closely enough to see that it is not.

**Open your output video and step through it frame by frame.** Scrubbing at normal speed hides everything — your eye smooths over a box that vanishes for two frames, but a planning system does not.

## The structural flaw

Every failure below traces to one design decision:

> **YOLO processes each frame completely independently.** Frame 100 has no memory of frame 99.

The model has no concept of an object *persisting through time*. Each frame it re-derives the world from scratch. A car detected in 40 consecutive frames is not "one car tracked for 40 frames" — it is 40 unrelated detections that happen to look similar, and nothing in the system knows they are the same vehicle.

---

## The three failure modes

### 1. Temporal flickering

A detection appears, vanishes for a frame or two, and returns. The object never moved.

**Why:** confidence hovers near your threshold. A tiny change — a pixel of motion blur, a lighting shift, a compression artifact — pushes it from 0.36 to 0.34 and the detection is gone. You drew a hard line through a continuous quantity.

**Why it matters:** at 30 FPS, three dropped frames is 100 ms. A car at 60 mph travels **2.7 metres** in that time. An emergency braking system that requires several consecutive confirmations before acting can be defeated by flicker alone.

### 2. Occlusion dropouts

A pedestrian walks behind a parked van and the detection disappears. When they emerge, a *new* detection appears — with no link to the old one.

**Why:** YOLO can only detect visible pixels. It has no object permanence — the understanding, which human infants develop around 8 months, that things continue to exist when hidden.

**Why it matters:** this is the single most dangerous gap. A child stepping behind a parked car is *predictably* about to reappear in the road. A per-frame detector reports "no pedestrian" and is, frame by frame, correct. The system needs to reason about what it *cannot currently see*, and a detector fundamentally cannot.

### 3. Adverse weather and lighting

Confidence collapses in rain, fog, snow, at night, and into low sun.

**Why:** COCO is mostly well-lit photographs taken by people who wanted good pictures. Almost nobody photographs a rainy motorway at 3 a.m. The model never learned those conditions — and it fails *silently*, reporting low confidence or nothing rather than "I cannot see."

**Why it matters:** this is Phase 1's brittleness returning in a new form. Your HSV thresholds failed on conditions *you* did not anticipate; YOLO fails on conditions the *dataset* did not cover. Learning moved the problem from your imagination to your data collection. It did not remove it.

---

## The thread through all three phases

| | Phase 1 | Phase 2 | Phase 3 |
|---|---|---|---|
| Features | hand-designed | learned | learned |
| Location | ✅ contours | ❌ flattened away | ✅ grid preserved |
| Speed | fast | fast | fast |
| Hand-tuned knobs | ~10 HSV/shape | learning rate, epochs | `conf`, `iou`, class list |
| Coverage limited by | your imagination | your training data | your training data |
| **Time** | ❌ none | ❌ none | ❌ **none** |

Each phase solved the previous one's headline problem and inherited a subtler one. The remaining gap is the entire bottom row: **nothing you have built so far has any concept of time.**

That is what tracking adds — and it is what makes questions like *"is that pedestrian moving toward the road?"* answerable at all.

### Exercise 4.1 — Frame-by-frame inspection

Use this stepper to hunt for failures. Move **one frame at a time** through any interval where the count plot in Exercise 3.3 showed a spike — that is where the bugs are.

In [ ]:
def frame_stepper(frame_number):
    cap = cv2.VideoCapture(OUTPUT_VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number - 1)
    ok, frame = cap.read()
    cap.release()

    if not ok:
        print("Could not read that frame.")
        return

    plt.figure(figsize=(11, 6.5))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f"annotated frame {frame_number}")
    plt.axis("off")
    plt.show()

    if frame_number <= len(per_frame_counts):
        c = per_frame_counts[frame_number - 1]
        print("this frame :", dict(c) or "nothing detected")
        if frame_number > 1:
            prev = per_frame_counts[frame_number - 2]
            gained = {k: v for k, v in c.items() if v > prev.get(k, 0)}
            lost = {k: prev[k] for k in prev if prev[k] > c.get(k, 0)}
            if gained or lost:
                print(f"CHANGED from previous frame — appeared: {gained or '{}'}, "
                      f"disappeared: {lost or '{}'}")


n_out = len(per_frame_counts) if per_frame_counts else 1
if os.path.exists(OUTPUT_VIDEO) and n_out > 1:
    display(widgets.interactive(
        frame_stepper,
        frame_number=widgets.IntSlider(value=1, min=1, max=n_out, step=1,
                                       description="frame", continuous_update=False),
    ))
else:
    print("Run Exercise 3.2 first to produce the annotated video.")

### Exercise 4.2 — Confidence sensitivity

Before writing up, get evidence for the flickering claim. Re-run detection on a handful of frames at several confidence thresholds and see how many detections sit right on the boundary.

Detections that vanish between 0.30 and 0.40 are precisely the ones that flicker in your video.

In [ ]:
if HAS_YOLO and model is not None:
    sample_idxs = np.linspace(0, max(src_n - 1, 0), 12, dtype=int)
    levels = [0.15, 0.25, 0.35, 0.50, 0.70]

    cap = cv2.VideoCapture(INPUT_VIDEO)
    frames = []
    for idx in sample_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, f = cap.read()
        if ok:
            frames.append(f)
    cap.release()

    totals = []
    for conf in levels:
        n = sum(
            len(model(f, conf=conf, iou=IOU_THRESHOLD, classes=ROAD_CLASS_IDS,
                      verbose=False)[0].boxes)
            for f in frames
        )
        totals.append(n)

    plt.figure(figsize=(9, 4.5))
    plt.plot(levels, totals, "o-")
    plt.axvline(CONF_THRESHOLD, ls="--", c="red", label=f"your threshold ({CONF_THRESHOLD})")
    plt.xlabel("confidence threshold")
    plt.ylabel(f"total detections over {len(frames)} sampled frames")
    plt.title("Detections vs. confidence threshold")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    for c, n in zip(levels, totals):
        print(f"  conf >= {c:.2f}  ->  {n:>4} detections")
    print("\nThe steeper this curve near your threshold, the more detections are")
    print("balanced on a knife edge — and those are the ones that flicker.")
else:
    print("Needs a loaded YOLO model.")

### Exercise 4.3 — Build your own adverse conditions

You probably do not have dashcam footage in fog. Simulate it: degrade a real frame and watch confidence fall.

These effects are crude approximations — real rain has streaks, real night has glare and rolling-shutter smear, real snow occludes. The true failures are worse than what you will see here.

In [ ]:
def degrade(frame, mode, rng=np.random.default_rng(0)):
    f = frame.copy()
    if mode == "clean":
        return f
    if mode == "fog":
        return cv2.addWeighted(f, 0.45, np.full_like(f, 200), 0.55, 0)
    if mode == "night":
        dark = (f * 0.28).astype(np.uint8)
        return cv2.add(dark, rng.integers(0, 22, f.shape, dtype=np.uint8))
    if mode == "rain":
        f = cv2.GaussianBlur(f, (5, 5), 0)
        for _ in range(700):
            x, y = rng.integers(0, f.shape[1]), rng.integers(0, f.shape[0] - 18)
            cv2.line(f, (x, y), (x - 2, y + rng.integers(8, 18)), (210, 210, 215), 1)
        return f
    if mode == "low sun":
        glare = np.zeros_like(f)
        cv2.circle(glare, (int(f.shape[1] * 0.7), int(f.shape[0] * 0.3)),
                   int(f.shape[1] * 0.3), (255, 245, 225), -1)
        glare = cv2.GaussianBlur(glare, (151, 151), 0)
        return cv2.addWeighted(f, 0.75, glare, 0.65, 0)
    if mode == "motion blur":
        k = np.zeros((17, 17), np.float32)
        k[8, :] = 1 / 17
        return cv2.filter2D(f, -1, k)
    return f


MODES = ["clean", "fog", "night", "rain", "low sun", "motion blur"]

if HAS_YOLO and model is not None:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    summary = []

    for ax, mode in zip(axes.flat, MODES):
        img = degrade(first_frame, mode)
        res = model(img, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                    classes=ROAD_CLASS_IDS, verbose=False)[0]
        n = len(res.boxes)
        mean_conf = float(res.boxes.conf.mean()) if n else 0.0
        summary.append((mode, n, mean_conf))

        ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{mode} — {n} detections, mean conf {mean_conf:.2f}", fontsize=10)
        ax.axis("off")

    plt.suptitle("Same frame, six conditions", fontsize=14)
    plt.tight_layout()
    plt.show()

    base = summary[0][1]
    print(f"{'condition':<14} {'detections':>11} {'mean conf':>11} {'vs clean':>10}")
    print("-" * 50)
    for mode, n, mc in summary:
        delta = f"{n - base:+d}" if mode != "clean" else "—"
        print(f"{mode:<14} {n:>11} {mc:>11.2f} {delta:>10}")
else:
    print("Needs a loaded YOLO model.")

---
## ✍️ Final Reflection — Write-up

Answer in full sentences, citing **specific frame numbers and timestamps** from your own video. Vague answers earn no credit — the skill being assessed is precise observation of a system's failures.

---

**1. Temporal flickering.**
Find one detection that disappears and reappears while the object is plainly still there. Give the frame numbers. Using Exercise 4.2, explain what its confidence was likely doing around your threshold. Then calculate: at 30 FPS, how far does a vehicle at 60 mph (27 m/s) travel during your observed dropout? Is that distance safe?

**2. Occlusion dropout.**
Find a frame range where an object passes behind something. What happened to its detection? When it reappeared, did the system have any way of knowing it was the *same* object? Explain why object permanence cannot be fixed by better training or a bigger model — say what would actually be required.

**3. Adverse conditions.**
Using your Exercise 4.3 table, rank the six conditions from least to most damaging. For the worst one, explain *why* it breaks the model in terms of what COCO contains. Then: the model did not announce that it was struggling — it simply reported fewer detections. Why is a detector that fails *silently* more dangerous than one that raises an error?

**4. The threshold trade-off.**
You set `CONF_THRESHOLD` and `IOU_THRESHOLD` by hand. For each, describe what goes wrong if it is too low and what goes wrong if it is too high, with a driving consequence for each of the four cases. Then argue: is there a single pair of values that is correct for both an empty motorway and a crowded school crossing?

**5. The missing dimension.**
Complete this sentence and defend it in a paragraph: *"YOLO tells the car what is in the frame and where it is, but it cannot tell the car ______, because ______."* Reference the fact that frames are processed independently.

**6. Three phases, one thread.**
Phase 1 failed on conditions you did not anticipate. Phase 3 fails on conditions the dataset did not cover. Are these the same problem or different problems? Defend your answer. Then: which knobs in this notebook are *still* hand-tuned, and what does their survival tell you about how much of engineering deep learning is actually deep learning?

**7. Would you ship it?**
Your detector works on your clip. Write a short, honest paragraph to a hypothetical safety engineer explaining why it must not be deployed. Name at least four specific limitations, and state what evidence you would need before changing your answer.

*Your write-up:*

**1.**

**2.**

**3.**

**4.**

**5.**

**6.**

**7.**

---
## Stretch goals (optional)

- **Add tracking.** Swap `model(frame)` for `model.track(frame, persist=True)`. Each object gains a persistent ID. Re-run Exercise 3.3 — does the count plot get smoother? Watch what happens to IDs across an occlusion.
- **Temporal smoothing by hand.** Keep a detection alive for up to 5 frames after it disappears, and require 3 consecutive frames before showing a new one. Measure the flicker percentage before and after. What new risk did you just introduce? (Hint: what if the object *really* vanished?)
- **Model size comparison.** Run `yolov8s.pt` and `yolov8m.pt` on the same clip. Plot accuracy against FPS. Where is the point past which a bigger model is too slow to be safe?
- **Estimate distance.** Assuming a pedestrian is ~1.7 m tall, use box height and a rough focal length to estimate range. Compare your estimates across frames as someone approaches. How badly does it fail for a seated or partly occluded person?
- **Region of interest.** Ignore detections outside the drivable area ahead. How many false alarms does this remove — and what does it cost you when a child runs in from the pavement?
- **Build a flicker metric.** Write a function counting how often each class's count changes between consecutive frames. Use it to compare `conf` values quantitatively instead of by eye.

---

### ➡️ Where this goes next
**Tracking** gives objects identity across time. **Sensor fusion** adds radar and lidar so rain and darkness do not blind the whole system. **Prediction** asks where each tracked object will be in three seconds. **Planning** turns all of it into steering and braking.

You have built the perception front-end. Everything downstream depends on it being honest about what it does not know.